In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from keras.datasets import cifar10
from keras.utils import to_categorical
from tensorflow.keras.applications import Xception, InceptionV3
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, concatenate, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf


In [2]:
# Load CIFAR-10 dataset
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

# Split into training and validation sets
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.1, random_state=0)

X_train = X_train.astype('float32')
X_test  = X_test.astype('float32')
X_valid = X_valid.astype('float32')

X_train /= 255.0
X_test  /= 255.0
X_valid /= 255.0

# One-hot encoding of labels
y_train = to_categorical(y_train, 10)
y_valid = to_categorical(y_valid, 10)
y_test  = to_categorical(y_test, 10)

# Resize inputs for the models
X_train_resized = tf.image.resize(X_train, [75, 75])
X_valid_resized = tf.image.resize(X_valid, [75, 75])
X_test_resized  = tf.image.resize(X_test, [75, 75])


170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [3]:
# Enable mixed precision training
from tensorflow.keras.mixed_precision import set_global_policy
set_global_policy('mixed_float16')

In [4]:
# Hyperparameters
batch_size = 128
input_shape = (75, 75, 3)  # Unified input size for both models
learning_rate = 0.001

def extract_features(base_model, data, batch_size):
    """Extract features using a pre-trained base model."""
    feature_model = Model(inputs=base_model.input, outputs=base_model.output)
    features = feature_model.predict(data, batch_size=batch_size, verbose=1)
    return features

In [5]:
# Pre-trained model for Xception
xception_base = Xception(weights='imagenet', include_top=False, input_shape=input_shape)
xception_output = GlobalAveragePooling2D()(xception_base.output)
xception_model = Model(inputs=xception_base.input, outputs=xception_output)

# Pre-trained model for InceptionV3
inception_base = InceptionV3(weights='imagenet', include_top=False, input_shape=input_shape)
inception_output = GlobalAveragePooling2D()(inception_base.output)
inception_model = Model(inputs=inception_base.input, outputs=inception_output)

# Freeze layers in base models
for layer in xception_base.layers:
    layer.trainable = False
for layer in inception_base.layers:
    layer.trainable = False

# Data Generators with Augmentation
data_generator = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

# Use resized inputs for the data generator
train_generator = data_generator.flow(X_train_resized, y_train, batch_size=batch_size)
valid_generator = data_generator.flow(X_valid_resized, y_valid, batch_size=batch_size)

# Extract Features using resized data
xception_features_train = extract_features(xception_model, X_train_resized, batch_size)
xception_features_valid = extract_features(xception_model, X_valid_resized, batch_size)
xception_features_test  = extract_features(xception_model, X_test_resized, batch_size)

inception_features_train = extract_features(inception_model, X_train_resized, batch_size)
inception_features_valid = extract_features(inception_model, X_valid_resized, batch_size)
inception_features_test  = extract_features(inception_model, X_test_resized, batch_size)

# Combine Features
combined_features_train = np.concatenate([xception_features_train, inception_features_train], axis=1)
combined_features_valid = np.concatenate([xception_features_valid, inception_features_valid], axis=1)
combined_features_test  = np.concatenate([xception_features_test,  inception_features_test], axis=1)

# Build the Top Model
top_model_input = tf.keras.Input(shape=combined_features_train.shape[1:])
x = Dense(512, activation='relu')(top_model_input)
x = Dropout(0.5)(x)
output = Dense(10, activation='softmax', dtype='float32')(x)

final_model = Model(inputs=top_model_input, outputs=output)

# Compile the Model
optimizer = Adam(learning_rate=learning_rate)
final_model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Callbacks
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5, verbose=1)
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
352/352 ━━━━━━━━━━━━━━━━━━━━ 3513s 10s/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 390s 10s/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 773s 10s/step
352/352 ━━━━━━━━━━━━━━━━━━━━ 582s 2s/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 69s 2s/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 133s 2s/step


In [9]:
epochs=30

# Train the Top Model
final_model.fit(
    combined_features_train, y_train,
    validation_data=(combined_features_valid, y_valid),
    batch_size=batch_size,
    epochs=epochs,
    callbacks=[reduce_lr, early_stopping],
    verbose=1
)

Epoch 1/30
352/352 ━━━━━━━━━━━━━━━━━━━━ 21s 59ms/step - accuracy: 0.7346 - loss: 0.7689 - val_accuracy: 0.7632 - val_loss: 0.6884 - learning_rate: 0.0010
Epoch 2/30
352/352 ━━━━━━━━━━━━━━━━━━━━ 24s 67ms/step - accuracy: 0.7669 - loss: 0.6661 - val_accuracy: 0.7612 - val_loss: 0.6782 - learning_rate: 0.0010
Epoch 3/30
352/352 ━━━━━━━━━━━━━━━━━━━━ 38s 60ms/step - accuracy: 0.7919 - loss: 0.5924 - val_accuracy: 0.7656 - val_loss: 0.6678 - learning_rate: 0.0010
Epoch 4/30
352/352 ━━━━━━━━━━━━━━━━━━━━ 40s 58ms/step - accuracy: 0.8074 - loss: 0.5452 - val_accuracy: 0.7732 - val_loss: 0.6649 - learning_rate: 0.0010
Epoch 5/30
352/352 ━━━━━━━━━━━━━━━━━━━━ 22s 61ms/step - accuracy: 0.8230 - loss: 0.4960 - val_accuracy: 0.7704 - val_loss: 0.6746 - learning_rate: 0.0010
Epoch 6/30
352/352 ━━━━━━━━━━━━━━━━━━━━ 41s 60ms/step - accuracy: 0.8416 - loss: 0.4431 - val_accuracy: 0.7720 - val_loss: 0.6857 - learning_rate: 0.0010
Epoch 7/30
352/352 ━━━━━━━━━━━━━━━━━━━━ 41s 60ms/step - accuracy: 0.8534 - l

In [12]:
# Evaluate the model on test data
test_loss, test_acc = final_model.evaluate(combined_features_test, y_test, verbose=1)

print(f'\nTest Accuracy: {test_acc:.4f}')
print(f'Test Loss: {test_loss:.4f}')

313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.7541 - loss: 0.7139

Test Accuracy: 0.7605
Test Loss: 0.7034


In [13]:
pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.6 MB/s eta 0:00:00
  Attempting uninstall: markupsafe
    Found existing installation: MarkupSafe 3.0.2
    Uninstalling MarkupSafe-3.0.2:
      Successfully uninstalled MarkupSafe-3.0.2


In [42]:
import gradio as gr
from tensorflow.keras.applications.xception import preprocess_input
from PIL import Image
import numpy as np

# Preprocess function
def preprocess_image(image):
    """Preprocess the input image for prediction."""
    image = image.resize((75, 75))  # Resize to match model input
    image = np.array(image).astype("float32") / 255.0  # Normalize
    image = np.expand_dims(image, axis=0)  # Add batch dimension
    return image

# Prediction function
def classify_image(image):
    """Predict the class of the input image."""
    # Preprocess the image
    preprocessed_image = preprocess_image(image)

    # Step 1: Extract features using Xception
    xception_features = xception_model.predict(preprocessed_image)
    # Debugging output
    print(f"Xception Features Shape: {xception_features.shape}")

    # Step 2: Extract features using Inception
    inception_features = inception_model.predict(preprocessed_image)
    # Debugging output
    print(f"Inception Features Shape: {inception_features.shape}")

    # Step 3: Combine features
    combined_features = np.concatenate([xception_features, inception_features], axis=1)
    # Debugging output
    print(f"Combined Features Shape: {combined_features.shape}")

    # Step 4: Make prediction
    predictions = final_model.predict(combined_features)
    class_index = np.argmax(predictions)

    # CIFAR-10 class labels
    class_labels = [
        "Airplane", "Automobile", "Bird", "Cat",
        "Deer", "Dog", "Frog", "Horse", "Ship", "Truck"
    ]

    # Debugging output
    print(f"Predictions: {predictions}")

    return {class_labels[class_index]}

# Create Gradio interface
interface = gr.Interface(
    fn=classify_image,
    inputs=gr.Image(type="pil"),
    outputs="text",
    title="CIFAR-10 Image Classifier",
    description="Upload an image to classify it into one of the 10 CIFAR-10 categories: Airplane, Automobile, Bird, Cat, Deer, Dog, Frog, Horse, Ship, Truck."
)

# Launch the Gradio interface
interface.launch()


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ae3531c3750b7a28e4.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
